Reviewer is concerned about the quality of assembly.
Here, based on their suggestions, I remapped the reads to manually curated genomes.

Procedure:
1. Starting from fastq used for assembly ...
2. filtlong
3. guppy trim (only the ends)
4. cutadapt (for internal adapters)
5. extract reads with clipping
6. Output the read depth and clipping positions
7. (In another R script) visualize the results

Note: fastq files are not tracked by git.
Fastq of files excluding L07-4_G8 are as previous.
Fastq of L07-4_G8 was replaced as below.
```bash
rm -r ../data/fastq/L07-4_G8
rsync -ahv ../../dat/fix_assembly_err/fastq/dat/L07-4_G8 ../data/fastq/
rsync -ahv ../../dat/fix_assembly_err/fastq/dat/L11_Anc ../data/fastq/
```

In [1]:
import pandas as pd
import re, io, subprocess
import os, sys

from Bio import SeqIO
from Bio.Seq import Seq

fastq_dir = '../data/fastq'
genome_fasta_dir = '../data/ref_fasta_run_14'
contaminate_fasta_dir = '../data/contaminations_in_fastq'
tmp_dir = '../tmp'
bam_exp_dir = '../exp/bam'

In [2]:
ids = pd.read_csv('../data/File_list_20250204.csv')
# convert the dataframe compatible to previous runs
ids = ids[ids.gen != 'Anc'] # remove MDS42
ids['Fasta'] = ids['File']
ids['Sample_2'] = ids['sample_name_raw']
ids['Sample'] = ids['IS_Detect_ID']
ids = ids.drop_duplicates(subset=['Sample'])
ids.head()

,IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File,Fasta,Sample_2,Sample
1,L01_Anc,1,1,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-1,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta,L01_Anc_m1.fasta,L01_Anc,L01_Anc
5,L02_Anc,2,1,FACS,20231003/L02_Anc.fasta,R02,1,L02-1,L02_Anc,20220418,True,20231003.0,20231003.0,NaN,L02_Anc.fasta,L02_Anc.fasta,L02_Anc,L02_Anc
9,L03_Anc,3,1,FACS,20240703/L03_Anc_m1.fasta,R03,1,L03-1,L03_Anc,20220418,True,20240703.0,20240703.0,NaN,L03_Anc_m1.fasta,L03_Anc_m1.fasta,L03_Anc,L03_Anc
13,L04_Anc,4,1,FACS,20231004/L04_Anc_m1.fasta,R04,1,L04-1,L04_Anc,20230717,True,20231004.0,20231004.0,NaN,L04_Anc_m1.fasta,L04_Anc_m1.fasta,L04_Anc,L04_Anc
17,L05_Anc,5,1,FACS,20231004/L05_Anc_m1.fasta,R05,1,L05-1,L05_Anc,20230909,True,20231004.0,20231004.0,NaN,L05_Anc_m1.fasta,L05_Anc_m1.fasta,L05_Anc,L05_Anc


In [3]:
contamination_ids = pd.read_csv(os.path.join(contaminate_fasta_dir, 'contaminations.tsv'), sep='\t')
contamination_ids.head()

,Name,Sample,Line,Subline,Gen
0,L01-1_G8_contig_1,L01-1_G08,1,1,8
1,L01-1_G8_contig_3,L01-1_G08,1,1,8
2,L02-1_G8_contig_1,L02-1_G08,2,1,8
3,YK_plasmid_3B2,L02-1_G08,2,1,8


## Prepare target fasta

In [71]:
def rebase_circular_fasta_with_blast(fasta_file, output_file, anchor_sequence, blast_dir):
    log_dir = os.path.join(blast_dir, "log")
    stdout_file = os.path.join(log_dir, "stdout.txt")
    stderr_file = os.path.join(log_dir, "stderr.txt")
    
    # Step 1: Create a BLAST database
    makeblastdb_cmd = f'makeblastdb -in {fasta_file} -dbtype nucl -out {blast_dir}/temp_db'
    #subprocess.run(makeblastdb_cmd, check=True, stdout=stdout_file, stderr=stderr_file)
    print(makeblastdb_cmd)
    subprocess.Popen(makeblastdb_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE).communicate()

    # Step 2: Write the anchor sequence to a temporary query file
    query_dir = os.path.join(blast_dir, "query.fasta")
    with open(query_dir, "w") as query_file:
        query_file.write(f">anchor_query\n{Seq(anchor_sequence)}\n")

    # Step 3: Run BLAST
    blast_cmd = f'blastn -query {query_dir} -db {blast_dir}/temp_db -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore sstrand" -max_target_seqs 1 -strand both'
    blast_cmd += f' -out {blast_dir}/blast_result.txt'
    print(blast_cmd)
    subprocess.Popen(blast_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE).communicate()


    # if empty skip
    blast_output_dir = f"{blast_dir}/blast_result.txt"
    blast_df = pd.DataFrame()
    try:
        blast_df = pd.read_csv(blast_output_dir, sep="\t", header=None)
    except:
        print("Anchor sequence not found in the genome!")
        return False
    # use the best match as anchor
    blast_df.columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore", "sstrand"]
    reverse = blast_df.iloc[0,:].sstart > blast_df.iloc[0,:].send
    print(blast_df)

    # Step 4: Load the genome sequence and rebase
    record = SeqIO.read(fasta_file, "fasta")
    sequence = str(record.seq)

    if not reverse:
        anchor_pos = blast_df.iloc[0].sstart + blast_df.iloc[0].qstart - 1
        rebased_sequence = sequence[anchor_pos - 1:] + sequence[:anchor_pos - 1]
    else:
        anchor_pos = blast_df.iloc[0].sstart - blast_df.iloc[0].qstart + 1
        rebased_sequence = sequence[anchor_pos:] + sequence[:anchor_pos]
        rebased_sequence = str(Seq(rebased_sequence).reverse_complement())

    # Step 6: Write the rebased sequence to the output file
    record.seq = Seq(rebased_sequence)
    record.description += " | Rebasing complete using BLAST"
    record.description += f" in reverse" if reverse else f" in forward" 
    record.description += f" direction"
    SeqIO.write(record, output_file, "fasta")
    
    # Step 7: Clean up temporary files
    for temp_file in ["temp_db.nhr", "temp_db.nin", "temp_db.nsq"]:
        temp_file = os.path.join(blast_dir, temp_file)
        subprocess.run(["rm", temp_file])
    
    print(f"Rebased sequence saved to {output_file}, reverse: {reverse}")
    return True

In [74]:
blast_dir_base = f'{tmp_dir}/blast'
rebase_dir = f'{tmp_dir}/rebased'
ori_seq = 'gatctatttatttagagatctgttctattgtgatctcttattaggatcgcactgccctgt'
os.makedirs(blast_dir_base, exist_ok=True)
os.makedirs(rebase_dir, exist_ok=True)

#for row in ids.query('Sample == "L03_Anc"').itertuples():
for row in ids.itertuples():
	print(row.Sample)
	blast_dir = os.path.join(blast_dir_base, f'{row.Sample}')
	os.makedirs(blast_dir, exist_ok=True)
	raw_fasta = os.path.join(genome_fasta_dir, f'{row.Fasta}')
	rebased_fasta = os.path.join(rebase_dir , f'{row.Sample}.fasta')
	
	rebased_ = rebase_circular_fasta_with_blast(raw_fasta, rebased_fasta, ori_seq, blast_dir)
	if not rebased_:
		print(f"Failed to rebase {row.Sample}")


L01_Anc
makeblastdb -in ../data/ref_fasta_run_14/L01_Anc_m1.fasta -dbtype nucl -out ../tmp/blast/L01_Anc/temp_db
blastn -query ../tmp/blast/L01_Anc/query.fasta -db ../tmp/blast/L01_Anc/temp_db -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore sstrand" -max_target_seqs 1 -strand both -out ../tmp/blast/L01_Anc/blast_result.txt
         qseqid      sseqid  pident  length  mismatch  gapopen  qstart  qend  \
0  anchor_query  L01_Anc_m1   100.0      60         0        0       1    60   

   sstart  send        evalue  bitscore sstrand  
0       1    60  3.340000e-26       111    plus  
Rebased sequence saved to ../tmp/rebased/L01_Anc.fasta, reverse: False
L02_Anc
makeblastdb -in ../data/ref_fasta_run_14/L02_Anc.fasta -dbtype nucl -out ../tmp/blast/L02_Anc/temp_db
blastn -query ../tmp/blast/L02_Anc/query.fasta -db ../tmp/blast/L02_Anc/temp_db -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore sstrand" 

In [75]:
# read fasta files and change it so that the contig names would be the same as the sample names
# create target for mm2
target_fasta_dir = f"{tmp_dir}/clean_fasta"
#rm files
subprocess.run(f"rm -rf {target_fasta_dir}", shell=True)
os.makedirs(target_fasta_dir, exist_ok=True)

import os
from Bio import SeqIO

for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
    #fasta_path = os.path.join(genome_fasta_dir, row.Fasta)
    fasta_path = os.path.join(rebase_dir, f"{row.Sample}.fasta")
    output_path = os.path.join(target_fasta_dir, f"{row.Sample}.fasta")

    # Check if the main fasta exists before processing
    if os.path.exists(fasta_path):
        # Open the original fasta and modify contig names
        with open(fasta_path, "r") as fasta_file, open(output_path, "w") as output_file:
            for record in SeqIO.parse(fasta_file, "fasta"):
                record.id = row.Sample
                record.description = ""  # Clear description for simplicity
                SeqIO.write(record, output_file, "fasta")
    else:
        print(f"File {fasta_path} not found")
        continue

    # Append contamination FASTA files if they exist
    contaminations = contamination_ids[contamination_ids.Sample == row.Sample]
    for contamination in contaminations.itertuples():
        contaminate_fasta_path = os.path.join(contaminate_fasta_dir, 'contaminate_plasmid', f'{contamination.Name}.fasta')
        if os.path.exists(contaminate_fasta_path):
            with open(contaminate_fasta_path, "r") as contaminate_fasta_file, open(output_path, "a") as output_file:
                for record in SeqIO.parse(contaminate_fasta_file, "fasta"):
                    SeqIO.write(record, output_file, "fasta")
            print(f"Appended {contaminate_fasta_path} to {output_path}") 
        else:
            print(f"File {contaminate_fasta_path} not found")
                    
    print(f"Processed {fasta_path} -> {output_path}")

Processed ../tmp/rebased/L01_Anc.fasta -> ../tmp/clean_fasta/L01_Anc.fasta
Processed ../tmp/rebased/L02_Anc.fasta -> ../tmp/clean_fasta/L02_Anc.fasta
Processed ../tmp/rebased/L03_Anc.fasta -> ../tmp/clean_fasta/L03_Anc.fasta
Processed ../tmp/rebased/L04_Anc.fasta -> ../tmp/clean_fasta/L04_Anc.fasta
Processed ../tmp/rebased/L05_Anc.fasta -> ../tmp/clean_fasta/L05_Anc.fasta
Processed ../tmp/rebased/L06_Anc.fasta -> ../tmp/clean_fasta/L06_Anc.fasta
Processed ../tmp/rebased/L07_Anc.fasta -> ../tmp/clean_fasta/L07_Anc.fasta
Processed ../tmp/rebased/L08_Anc.fasta -> ../tmp/clean_fasta/L08_Anc.fasta
Processed ../tmp/rebased/L09_Anc.fasta -> ../tmp/clean_fasta/L09_Anc.fasta
Processed ../tmp/rebased/L10_Anc.fasta -> ../tmp/clean_fasta/L10_Anc.fasta
Processed ../tmp/rebased/L11_Anc.fasta -> ../tmp/clean_fasta/L11_Anc.fasta
Appended ../data/contaminations_in_fastq/contaminate_plasmid/L01-1_G8_contig_1.fasta to ../tmp/clean_fasta/L01-1_G08.fasta
Appended ../data/contaminations_in_fastq/contaminate

## Prepare read fastq
1. filter short reads
2. adapter trimming (ends)
3. adapter trimming (internal)

In [23]:
# trim barcodes and adapters i forgot to do for the older samples
output_dir = f'{tmp_dir}/fastq_filtlong'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{tmp_dir}/concatenated_fastq', exist_ok=True)

log_dir = os.path.join(output_dir, 'logs')
os.makedirs(log_dir, exist_ok=True)
processes = []
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
	#print(row)
	sample = row.Sample
	concatenated_fastq = f'{tmp_dir}/concatenated_fastq/{sample}.fastq.gz'
	sample_2 = row.Sample_2 # old sample name without zero padding
	raw_fastq_dir = os.path.join(fastq_dir, sample_2)
	C = f'find {raw_fastq_dir} -name "*.fastq.gz" -exec cat ' + '{} + > ' + concatenated_fastq
	print(C)
	subprocess.run(C, shell=True)

	# run filtlong
	# keep ~ x50 coverage
	output_dir_sample = os.path.join(output_dir, sample)
	os.makedirs(output_dir_sample, exist_ok=True)
	fqgz_path = os.path.join(output_dir_sample, f'{sample}.fastq.gz')

	C = f'filtlong --min_length 1000 --target_bases 200000000 --keep_percent 90 {concatenated_fastq} | pigz > {fqgz_path}'
	C += f' && rm {concatenated_fastq}'

	print(C)
	log_stdout = os.path.join(log_dir, f'{sample}.stdout')
	log_stderr = os.path.join(log_dir, f'{sample}.stderr')
	with open(log_stdout, 'w') as stdout_file, open(log_stderr, 'w') as stderr_file:
		p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
		processes.append(p)

	# parallel run
	if len(processes) > 5:
		for p in processes:
			p.wait()
		processes = []

for p in processes:
    p.wait()
processes = []

find ../data/fastq/L01_Anc -name "*.fastq.gz" -exec cat {} + > ../tmp/concatenated_fastq/L01_Anc.fastq.gz
filtlong --min_length 1000 --target_bases 200000000 --keep_percent 90 ../tmp/concatenated_fastq/L01_Anc.fastq.gz | pigz > ../tmp/fastq_filtlong/L01_Anc/L01_Anc.fastq.gz && rm ../tmp/concatenated_fastq/L01_Anc.fastq.gz
find ../data/fastq/L02_Anc -name "*.fastq.gz" -exec cat {} + > ../tmp/concatenated_fastq/L02_Anc.fastq.gz
filtlong --min_length 1000 --target_bases 200000000 --keep_percent 90 ../tmp/concatenated_fastq/L02_Anc.fastq.gz | pigz > ../tmp/fastq_filtlong/L02_Anc/L02_Anc.fastq.gz && rm ../tmp/concatenated_fastq/L02_Anc.fastq.gz
find ../data/fastq/L03_Anc -name "*.fastq.gz" -exec cat {} + > ../tmp/concatenated_fastq/L03_Anc.fastq.gz
filtlong --min_length 1000 --target_bases 200000000 --keep_percent 90 ../tmp/concatenated_fastq/L03_Anc.fastq.gz | pigz > ../tmp/fastq_filtlong/L03_Anc/L03_Anc.fastq.gz && rm ../tmp/concatenated_fastq/L03_Anc.fastq.gz
find ../data/fastq/L04_Anc -

In [6]:
# trim barcodes and adapters i forgot to do for the older samples
prev_output_dir = f'{tmp_dir}/fastq_filtlong'
output_dir = f'{tmp_dir}/fastq_trimmed'
log_dir = os.path.join(output_dir, 'logs')
os.makedirs(log_dir, exist_ok=True)

processes = []
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
	print(row)
	sample = row.Sample
	output_dir_sample = os.path.join(output_dir, sample)
	C = f'guppy_barcoder --input_path {prev_output_dir}/{sample} --save_path {output_dir_sample} --compress_fastq --device cuda:0 --enable_trim_barcodes --num_extra_bases_trim 2 --barcode_kits SQK-RBK110-96'
	# C += ' --detect_mid_strand_barcodes --detect_mid_strand_adapter ' # This just classifies reads to unclassified, use cut adapt instead
	print(C)
	log_stdout = os.path.join(log_dir, f'{sample}.stdout')
	log_stderr = os.path.join(log_dir, f'{sample}.stderr')
	with open(log_stdout, 'w') as stdout_file, open(log_stderr, 'w') as stderr_file:
		p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
		processes.append(p)

	if len(processes) > 7:
		for p in processes:
			p.wait()
		processes = []

for p in processes:
    p.wait()
processes = []

Pandas(Index=1, IS_Detect_ID='L01_Anc', ParentLine=1, SubLine=1, gen='FACS', file_name='20231004/L01_Anc_m1.fasta', Anc='R01', RecA=1, Prefix='L01-1', sample_name_raw='L01_Anc', Contig_Date='20220418', Complete=True, Folder=20231004.0, Folder_check=20231004.0, folder_err=nan, File='L01_Anc_m1.fasta', Fasta='L01_Anc_m1.fasta', Sample_2='L01_Anc', Sample='L01_Anc')
guppy_barcoder --input_path ../tmp/fastq_filtlong/L01_Anc --save_path ../tmp/fastq_trimmed/L01_Anc --compress_fastq --device cuda:0 --enable_trim_barcodes --num_extra_bases_trim 2 --barcode_kits SQK-RBK110-96
Pandas(Index=5, IS_Detect_ID='L02_Anc', ParentLine=2, SubLine=1, gen='FACS', file_name='20231003/L02_Anc.fasta', Anc='R02', RecA=1, Prefix='L02-1', sample_name_raw='L02_Anc', Contig_Date='20220418', Complete=True, Folder=20231003.0, Folder_check=20231003.0, folder_err=nan, File='L02_Anc.fasta', Fasta='L02_Anc.fasta', Sample_2='L02_Anc', Sample='L02_Anc')
guppy_barcoder --input_path ../tmp/fastq_filtlong/L02_Anc --save_pat

In [7]:
# remove chimeric reads with adapters
prev_output_dir = f'{tmp_dir}/fastq_trimmed'
output_dir = f'{tmp_dir}/fastq_cutadapt'
fastq_trimmed_concat_dir = f'{tmp_dir}/fastq_trimmed_concat'
os.makedirs(fastq_trimmed_concat_dir, exist_ok=True)

chimeric_reads = f'{tmp_dir}/chimeric_reads'
os.makedirs(chimeric_reads, exist_ok=True)

log_dir = os.path.join(output_dir, 'logs')
os.makedirs(log_dir, exist_ok=True)
processes = []


for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
	print(row)
	sample = row.Sample
	concatenated_fastq = f'{fastq_trimmed_concat_dir}/{sample}.fastq.gz'
	C = f'find {prev_output_dir}/{sample} -name "*.fastq.gz" -exec cat ' + '{} + > ' + concatenated_fastq
	print(C)
	subprocess.run(C, shell=True)

	# if adapters are detected sequence including the adapter would be trimmed and output to "-o"
	# sequence without adapter would be output to "--untrimmed-output"
	output_fastq_path = os.path.join(output_dir, f'{sample}.fastq.gz')
	C = f'cutadapt -o {chimeric_reads}/{sample}.fastq.gz --untrimmed-output  {output_fastq_path} '
	C += ' --cut 60 --cut -60 --minimum-length 1000 --action=lowercase '
	# detect common sequence in barcodes
	C += ' -g TTTTTTTTCCTGTACTTCGTTCAGTTACGTATTGCT -a AGCAATACGTAACTGAACGAAGTACAGGAAAAAAAA' # barcode
	# detect adapter sequence
	C += ' -g GTTTTCGCATTTATCGTGAAACGCTTTCGCGTTTTTCGTGCGCCGCTTCA -a TGAAGCGGCGCACGAAAAACGCGAAAGCGTTTCACGATAAATGCGAAAAC' # adapter
	C += ' --error-rate 0.25 --overlap 20 '
	C += concatenated_fastq

	print(C)
	log_stdout = os.path.join(log_dir, f'{sample}.stdout')
	log_stderr = os.path.join(log_dir, f'{sample}.stderr')
	with open(log_stdout, 'w') as stdout_file, open(log_stderr, 'w') as stderr_file:
		p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
		processes.append(p)

	if len(processes) >= 6:
		for p in processes:
			p.wait()
		processes = []

for p in processes:
    p.wait()
processes = []

Pandas(Index=1, IS_Detect_ID='L01_Anc', ParentLine=1, SubLine=1, gen='FACS', file_name='20231004/L01_Anc_m1.fasta', Anc='R01', RecA=1, Prefix='L01-1', sample_name_raw='L01_Anc', Contig_Date='20220418', Complete=True, Folder=20231004.0, Folder_check=20231004.0, folder_err=nan, File='L01_Anc_m1.fasta', Fasta='L01_Anc_m1.fasta', Sample_2='L01_Anc', Sample='L01_Anc')
find ../tmp/fastq_trimmed/L01_Anc -name "*.fastq.gz" -exec cat {} + > ../tmp/fastq_trimmed_concat/L01_Anc.fastq.gz
cutadapt -o ../tmp/chimeric_reads/L01_Anc.fastq.gz --untrimmed-output  ../tmp/fastq_cutadapt/L01_Anc.fastq.gz  --cut 60 --cut -60 --minimum-length 1000 --action=lowercase  -g TTTTTTTTCCTGTACTTCGTTCAGTTACGTATTGCT -a AGCAATACGTAACTGAACGAAGTACAGGAAAAAAAA -g GTTTTCGCATTTATCGTGAAACGCTTTCGCGTTTTTCGTGCGCCGCTTCA -a TGAAGCGGCGCACGAAAAACGCGAAAGCGTTTCACGATAAATGCGAAAAC --error-rate 0.25 --overlap 20 ../tmp/fastq_trimmed_concat/L01_Anc.fastq.gz
Pandas(Index=5, IS_Detect_ID='L02_Anc', ParentLine=2, SubLine=1, gen='FACS', file_n

# Run minimap2
run with trimmed reads and cutadapted reads

In [76]:
# cutadapt reads
fastq_input = f'{tmp_dir}/fastq_cutadapt'
mm2_ref_fasta_dir = f'{tmp_dir}/clean_fasta'

bamdir = f'{bam_exp_dir}/mm2_cutadapt/all'
os.makedirs(bamdir, exist_ok=True)
log_dir = os.path.join(bamdir, 'logs')
os.makedirs(log_dir, exist_ok=True)

processes = []
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
    sample = row.Sample
    # if output exists, skip
    bamfile = os.path.join(bamdir, f'{sample}.bam')
    log_stdout = os.path.join(log_dir, f'{sample}_stdout.log')
    log_stderr = os.path.join(log_dir, f'{sample}_stderr.log')
    fasta = f'{mm2_ref_fasta_dir}/{sample}.fasta'

    C = f'minimap2 -ax map-ont --secondary=no -L -t 1 {fasta} {fastq_input}/{sample}.fastq.gz '
    C += f' | samtools sort -@ 8 -m 2G -O BAM - > {bamfile}'
    C += f' && samtools index -@ 8 {bamfile}'
    print(C)

    with open(log_stdout, 'a') as stdout_file, open(log_stderr, 'a') as stderr_file:
        p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
        processes.append(p)

    if len(processes) > 6:
        for p in processes:
            p.wait()
        processes = []

for p in processes:
    p.wait()
processes = []

minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L01_Anc.fasta ../tmp/fastq_cutadapt/L01_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_cutadapt/all/L01_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_cutadapt/all/L01_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L02_Anc.fasta ../tmp/fastq_cutadapt/L02_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_cutadapt/all/L02_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_cutadapt/all/L02_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L03_Anc.fasta ../tmp/fastq_cutadapt/L03_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_cutadapt/all/L03_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_cutadapt/all/L03_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L04_Anc.fasta ../tmp/fastq_cutadapt/L04_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_cutadapt/all/L04_Anc.bam && samtools index -@ 8 ../exp/b

In [77]:
# trimmed reads
fastq_input = f'{tmp_dir}/fastq_trimmed_concat'
mm2_ref_fasta_dir = f'{tmp_dir}/clean_fasta'

bamdir = f'{bam_exp_dir}/mm2_trimmed/all'
os.makedirs(bamdir, exist_ok=True)
log_dir = os.path.join(bamdir, 'logs')
os.makedirs(log_dir, exist_ok=True)

processes = []
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
    sample = row.Sample
    # if output exists, skip
    bamfile = os.path.join(bamdir, f'{sample}.bam')
    log_stdout = os.path.join(log_dir, f'{sample}_stdout.log')
    log_stderr = os.path.join(log_dir, f'{sample}_stderr.log')
    fasta = f'{mm2_ref_fasta_dir}/{sample}.fasta'

    C = f'minimap2 -ax map-ont --secondary=no -L -t 1 {fasta} {fastq_input}/{sample}.fastq.gz '
    C += f' | samtools sort -@ 8 -m 2G -O BAM - > {bamfile}'
    C += f' && samtools index -@ 8 {bamfile}'
    print(C)

    with open(log_stdout, 'a') as stdout_file, open(log_stderr, 'a') as stderr_file:
        p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
        processes.append(p)

    if len(processes) > 6:
        for p in processes:
            p.wait()
        processes = []

for p in processes:
    p.wait()
processes = []

minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L01_Anc.fasta ../tmp/fastq_trimmed_concat/L01_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L01_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L01_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L02_Anc.fasta ../tmp/fastq_trimmed_concat/L02_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L02_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L02_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L03_Anc.fasta ../tmp/fastq_trimmed_concat/L03_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L03_Anc.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L03_Anc.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L04_Anc.fasta ../tmp/fastq_trimmed_concat/L04_Anc.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L04_Anc.bam && samtools in

minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L01-4_G08.fasta ../tmp/fastq_trimmed_concat/L01-4_G08.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L01-4_G08.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L01-4_G08.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L02-1_G08.fasta ../tmp/fastq_trimmed_concat/L02-1_G08.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L02-1_G08.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L02-1_G08.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L02-2_G08.fasta ../tmp/fastq_trimmed_concat/L02-2_G08.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/all/L02-2_G08.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/all/L02-2_G08.bam
minimap2 -ax map-ont --secondary=no -L -t 1 ../tmp/clean_fasta/L02-3_G08.fasta ../tmp/fastq_trimmed_concat/L02-3_G08.fastq.gz  | samtools sort -@ 8 -m 2G -O BAM - > ../exp/bam/mm2_trimmed/al

### Extract clipped reads and calculate depth

In [78]:
# cutadapt ver
bamdir_1 = f'{bam_exp_dir}/mm2_cutadapt/all'
bamdir_2 = f'{bam_exp_dir}/mm2_cutadapt/clipped'

log_dir = os.path.join(bamdir_2, 'logs')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(os.path.join(bamdir_1, 'depth'), exist_ok=True)
processes = []

for row in ids.itertuples():
    sample = row.Sample
    # if output exists, skip
    bamfile_1 = os.path.join(bamdir_1, f'{sample}.bam')
    bamfile_2 = os.path.join(bamdir_2, f'{sample}_clipped.bam')
    log_stdout = os.path.join(log_dir, f'{sample}_stdout.log')
    log_stderr = os.path.join(log_dir, f'{sample}_stderr.log')

    C = f'samtools view -H {bamfile_1} > {bamdir_2}/{sample}_header.txt && '
    C += f'samtools view {bamfile_1} | '
    C += 'awk \'$6 ~ /H|S/ {print}\''
    C += f' > {bamdir_2}/{sample}_clipped.txt && '
    C += f'cat {bamdir_2}/{sample}_header.txt {bamdir_2}/{sample}_clipped.txt | samtools view -bS > {bamfile_2}'
    C += f' && samtools index -@ 8 {bamfile_2}'
    C += f' && rm {bamdir_2}/{sample}_header.txt {bamdir_2}/{sample}_clipped.txt'
    # calculate full depth
    C += f' && samtools depth -aa {bamfile_1} > {bamdir_1}/depth/{sample}_depth.txt'

    print(C)

    with open(log_stdout, 'w') as stdout_file, open(log_stderr, 'w') as stderr_file:
        p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
        processes.append(p)

    if len(processes) > 7:
        for p in processes:
            p.wait()
        processes = []

for p in processes:
    p.wait()
processes = []

samtools view -H ../exp/bam/mm2_cutadapt/all/L01_Anc.bam > ../exp/bam/mm2_cutadapt/clipped/L01_Anc_header.txt && samtools view ../exp/bam/mm2_cutadapt/all/L01_Anc.bam | awk '$6 ~ /H|S/ {print}' > ../exp/bam/mm2_cutadapt/clipped/L01_Anc_clipped.txt && cat ../exp/bam/mm2_cutadapt/clipped/L01_Anc_header.txt ../exp/bam/mm2_cutadapt/clipped/L01_Anc_clipped.txt | samtools view -bS > ../exp/bam/mm2_cutadapt/clipped/L01_Anc_clipped.bam && samtools index -@ 8 ../exp/bam/mm2_cutadapt/clipped/L01_Anc_clipped.bam && rm ../exp/bam/mm2_cutadapt/clipped/L01_Anc_header.txt ../exp/bam/mm2_cutadapt/clipped/L01_Anc_clipped.txt && samtools depth -aa ../exp/bam/mm2_cutadapt/all/L01_Anc.bam > ../exp/bam/mm2_cutadapt/all/depth/L01_Anc_depth.txt
samtools view -H ../exp/bam/mm2_cutadapt/all/L02_Anc.bam > ../exp/bam/mm2_cutadapt/clipped/L02_Anc_header.txt && samtools view ../exp/bam/mm2_cutadapt/all/L02_Anc.bam | awk '$6 ~ /H|S/ {print}' > ../exp/bam/mm2_cutadapt/clipped/L02_Anc_clipped.txt && cat ../exp/bam/mm

In [79]:
# aggregate and compress depth
C = f'cat {bamdir_1}/depth/*_depth.txt | pigz > {bamdir_1}/depth/depth.txt.gz'
#C += f' && rm {bamdir_1}/depth/*_depth.txt'
print(C)
subprocess.run(C, shell=True)

cat ../exp/bam/mm2_cutadapt/all/depth/*_depth.txt | pigz > ../exp/bam/mm2_cutadapt/all/depth/depth.txt.gz


CompletedProcess(args='cat ../exp/bam/mm2_cutadapt/all/depth/*_depth.txt | pigz > ../exp/bam/mm2_cutadapt/all/depth/depth.txt.gz', returncode=0)

In [80]:
# trimmed ver
bamdir_1 = f'{bam_exp_dir}/mm2_trimmed/all'
bamdir_2 = f'{bam_exp_dir}/mm2_trimmed/clipped'

log_dir = os.path.join(bamdir_2, 'logs')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(os.path.join(bamdir_1, 'depth'), exist_ok=True)
processes = []

for row in ids.itertuples():
    sample = row.Sample
    # if output exists, skip
    bamfile_1 = os.path.join(bamdir_1, f'{sample}.bam')
    bamfile_2 = os.path.join(bamdir_2, f'{sample}_clipped.bam')
    log_stdout = os.path.join(log_dir, f'{sample}_stdout.log')
    log_stderr = os.path.join(log_dir, f'{sample}_stderr.log')

    C = f'samtools view -H {bamfile_1} > {bamdir_2}/{sample}_header.txt && '
    C += f'samtools view {bamfile_1} | '
    C += 'awk \'$6 ~ /H|S/ {print}\''
    C += f' > {bamdir_2}/{sample}_clipped.txt && '
    C += f'cat {bamdir_2}/{sample}_header.txt {bamdir_2}/{sample}_clipped.txt | samtools view -bS > {bamfile_2}'
    C += f' && samtools index -@ 8 {bamfile_2}'
    C += f' && rm {bamdir_2}/{sample}_header.txt {bamdir_2}/{sample}_clipped.txt'
    # calculate full depth
    C += f' && samtools depth -aa {bamfile_1} > {bamdir_1}/depth/{sample}_depth.txt'

    print(C)

    with open(log_stdout, 'w') as stdout_file, open(log_stderr, 'w') as stderr_file:
        p = subprocess.Popen(C, shell=True, stdin=None, stdout=stdout_file, stderr=stderr_file, close_fds=True)
        processes.append(p)

    if len(processes) > 7:
        for p in processes:
            p.wait()
        processes = []

for p in processes:
    p.wait()
processes = []

samtools view -H ../exp/bam/mm2_trimmed/all/L01_Anc.bam > ../exp/bam/mm2_trimmed/clipped/L01_Anc_header.txt && samtools view ../exp/bam/mm2_trimmed/all/L01_Anc.bam | awk '$6 ~ /H|S/ {print}' > ../exp/bam/mm2_trimmed/clipped/L01_Anc_clipped.txt && cat ../exp/bam/mm2_trimmed/clipped/L01_Anc_header.txt ../exp/bam/mm2_trimmed/clipped/L01_Anc_clipped.txt | samtools view -bS > ../exp/bam/mm2_trimmed/clipped/L01_Anc_clipped.bam && samtools index -@ 8 ../exp/bam/mm2_trimmed/clipped/L01_Anc_clipped.bam && rm ../exp/bam/mm2_trimmed/clipped/L01_Anc_header.txt ../exp/bam/mm2_trimmed/clipped/L01_Anc_clipped.txt && samtools depth -aa ../exp/bam/mm2_trimmed/all/L01_Anc.bam > ../exp/bam/mm2_trimmed/all/depth/L01_Anc_depth.txt
samtools view -H ../exp/bam/mm2_trimmed/all/L02_Anc.bam > ../exp/bam/mm2_trimmed/clipped/L02_Anc_header.txt && samtools view ../exp/bam/mm2_trimmed/all/L02_Anc.bam | awk '$6 ~ /H|S/ {print}' > ../exp/bam/mm2_trimmed/clipped/L02_Anc_clipped.txt && cat ../exp/bam/mm2_trimmed/clippe

In [81]:
# aggregate and compress depth
C = f'cat {bamdir_1}/depth/*_depth.txt | pigz > {bamdir_1}/depth/depth.txt.gz'
#C += f' && rm {bamdir_1}/depth/*_depth.txt'
print(C)
subprocess.run(C, shell=True)

cat ../exp/bam/mm2_trimmed/all/depth/*_depth.txt | pigz > ../exp/bam/mm2_trimmed/all/depth/depth.txt.gz


CompletedProcess(args='cat ../exp/bam/mm2_trimmed/all/depth/*_depth.txt | pigz > ../exp/bam/mm2_trimmed/all/depth/depth.txt.gz', returncode=0)

In [82]:
import pysam

# make below a function
def get_clipping_ends(bam_file, output_file, min_clip_length=100):
    # Open BAM file
    samfile = pysam.AlignmentFile(bam_file, "rb")

    # Prepare output file
    with open(output_file, "w") as out:
        out.write("Chromosome\tReadName\tPosition\tClippingType\tClippingLength\tReadDir\tEnd\tLength\tAlignedLength\n")

        for read in samfile:
            # Parse CIGAR string and extract clip information
            cigar = read.cigartuples
            start_pos = read.reference_start + 1
            end_pos = read.reference_end
            chrom = read.reference_name
            read_name = read.query_name
            read_length = read.query_length
            qa_length = read.query_alignment_length
            read_dir = "+" if not read.is_reverse else "-"

            # Check for soft/hard clipping and filter by length
            if cigar[0][0] in [4, 5]:  # Clipping at start
                clip_type = "S" if cigar[0][0] == 4 else "H"
                clip_length = cigar[0][1]
                if clip_length >= min_clip_length:
                    out.write(f"{chrom}\t{read_name}\t{start_pos}\t{clip_type}\t{clip_length}\t{read_dir}\tStart\t{read_length}\t{qa_length}\n")
            if cigar[-1][0] in [4, 5]:  # Clipping at end
                clip_type = "S" if cigar[-1][0] == 4 else "H"
                clip_length = cigar[-1][1]
                if clip_length >= min_clip_length:
                    out.write(f"{chrom}\t{read_name}\t{end_pos}\t{clip_type}\t{clip_length}\t{read_dir}\tEnd\t{read_length}\t{qa_length}\n")

    samfile.close()

In [83]:
# for cutadapt
bamdir_2 = f'{bam_exp_dir}/mm2_cutadapt/clipped'
out_dir = os.path.join(bamdir_2, 'clipped_ends')
os.makedirs(out_dir, exist_ok=True)
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
	sample = row.Sample
	bamfile = os.path.join(bamdir_2, f'{sample}_clipped.bam')
	if os.path.exists(bamfile):
		output_file = os.path.join(out_dir, f'filtered_clipped_ends_{sample}.txt')
		get_clipping_ends(bamfile, output_file)
		print(output_file)
	else:
		print(f"File {bamfile} not found")
		continue

../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L01_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L02_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L03_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L04_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L05_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L06_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L07_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L08_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L09_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L10_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L11_Anc.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filtered_clipped_ends_L01-1_G08.txt
../exp/bam/mm2_cutadapt/clipped/clipped_ends/filte

In [84]:
# for trimmed
bamdir_2 = f'{bam_exp_dir}/mm2_trimmed/clipped'
out_dir = os.path.join(bamdir_2, 'clipped_ends')
os.makedirs(out_dir, exist_ok=True)
for row in ids.drop_duplicates(subset='Sample').iloc[:].itertuples():
	sample = row.Sample
	bamfile = os.path.join(bamdir_2, f'{sample}_clipped.bam')
	if os.path.exists(bamfile):
		output_file = os.path.join(out_dir, f'filtered_clipped_ends_{sample}.txt')
		get_clipping_ends(bamfile, output_file)
		print(output_file)
	else:
		print(f"File {bamfile} not found")
		continue

../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L01_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L02_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L03_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L04_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L05_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L06_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L07_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L08_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L09_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L10_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L11_Anc.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_ends_L01-1_G08.txt
../exp/bam/mm2_trimmed/clipped/clipped_ends/filtered_clipped_e